# Soft Actor-Critic — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Reward plus entropy for robust exploration
SAC treats randomness as a feature of the objective, not merely training noise.

In [ ]:
# 1) softmax turns policy logits into action probabilities
logits=np.array([1.,0.]); p=softmax(logits)
plt.figure(figsize=(6,3)); plt.bar(['action 0','action 1'],p); plt.ylim(0,1); plt.title('softmax policy')
assert abs(p[0]-0.7310585786300049)<1e-12 and abs(p.sum()-1)<1e-12

In [ ]:
# 2) REINFORCE direction raises probability of a positively advantaged action
p=softmax([1.,0.]); A=0.7; grad=np.array([1-p[0], -p[1]])*A
plt.figure(figsize=(6,3)); plt.bar(['logit 0 grad','logit 1 grad'],grad); plt.title('positive advantage pushes chosen action up')
assert grad[0]>0 and grad[1]<0

In [ ]:
# 3) a value baseline reduces variance without changing the mean advantage much
rng=np.random.default_rng(1); returns=rng.normal(1.5,0.6,400); baseline=returns.mean(); adv=returns-baseline
plt.figure(figsize=(6,3)); plt.hist(returns,alpha=.5,label='returns'); plt.hist(adv,alpha=.5,label='advantages'); plt.legend(); plt.title('baseline centers the learning signal')
assert abs(adv.mean())<1e-12 and adv.std()<returns.std()+1e-12

In [ ]:
# 4) PPO clipping limits the incentive from a large probability ratio
r=np.linspace(0.5,1.8,100); A=1.0; eps=0.2; obj=np.minimum(r*A,np.clip(r,1-eps,1+eps)*A)
plt.figure(figsize=(6,3)); plt.plot(r,obj); plt.axvline(1+eps,ls='--',c='r'); plt.xlabel('probability ratio'); plt.ylabel('clipped objective'); plt.title('objective flattens beyond the trust region')
assert abs(obj[-1]-1.2)<1e-12

In [ ]:
# 5) entropy bonus favors a less collapsed policy
policies=np.array([[0.9,0.1],[0.55,0.45]]); H=-(policies*np.log(policies)).sum(axis=1)
plt.figure(figsize=(6,3)); plt.bar(['peaked','mixed'],H); plt.ylabel('entropy'); plt.title('SAC-style entropy rewards useful randomness')
assert H[1]>H[0]